In [1]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
from pathlib import Path

root = Path.cwd().parent
sys.path.append(str(root))
print(root)   # should end in Data-analysis

from src.ingestion.csv_ingestion import load_datasets
from src.cleaning.customers import clean_customers
from src.validation.data_quality import (
    check_missing_values, check_duplicates, check_unique_values, check_data_type,
)
from src.cleaning.orders import clean_orders, cols

c:\Users\USER\Desktop\data-analysis\Data-analysis


In [2]:
datasets = load_datasets()

In [3]:
name = "olist_customers_dataset"
raw = datasets[name]
clean = clean_customers(raw)
zip_col = "customer_zip_code_prefix"

# 1. raw table must be untouched
print("raw dtype (should be int64):", raw[zip_col].dtype)

# 2. nothing lost
print("rows raw/clean:", len(raw), len(clean))
print("columns equal:", list(raw.columns) == list(clean.columns))
print("distinct zips raw/clean:", raw[zip_col].nunique(), clean[zip_col].nunique())
print("other columns identical:", raw.drop(columns=zip_col).equals(clean.drop(columns=zip_col)))
print("round trip matches raw:", (clean[zip_col].astype(int) == raw[zip_col]).all())

# 3. the fix worked
print("clean dtype:", clean[zip_col].dtype)
print("lengths:\n", clean[zip_col].str.len().value_counts())

# 4. spot check: rows that lost a leading zero
changed = raw[raw[zip_col] < 10000].index[:5]
display(pd.DataFrame({"raw": raw.loc[changed, zip_col], "clean": clean.loc[changed, zip_col]}))

# 5. rerun the quality checks on the cleaned table
display(pd.concat([
    check_data_type(clean, name, zip_col, str),
    check_missing_values(clean, name),
    check_duplicates(clean, name),
    check_unique_values(clean, name, ["customer_id"]),
], ignore_index=True))

raw dtype (should be int64): int64
rows raw/clean: 99441 99441
columns equal: True
distinct zips raw/clean: 14994 14994
other columns identical: True
round trip matches raw: True
clean dtype: str
lengths:
 customer_zip_code_prefix
5    99441
Name: count, dtype: int64


,raw,clean
1,9790,09790
2,1151,01151
3,8775,08775
6,4534,04534
13,5704,05704


,table,check,column,count,percent,severity,message


In [5]:
orders = datasets["olist_orders_dataset"].copy()   # copy, raw stays untouched

for col in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
            "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[col] = pd.to_datetime(orders[col])

# null counts must still match: 0, 160, 1783, 2965, 0
print(orders.filter(like="order_").isnull().sum(), "\n")

for col in ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]:
    print(col)
    display(pd.crosstab(orders["order_status"], orders[col].isnull(), margins=True))

order_id                            0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64 

order_approved_at


order_approved_at,False,True,All
order_status,,,
approved,2,0,2
canceled,484,141,625
created,0,5,5
delivered,96464,14,96478
invoiced,314,0,314
processing,301,0,301
shipped,1107,0,1107
unavailable,609,0,609
All,99281,160,99441


order_delivered_carrier_date


order_delivered_carrier_date,False,True,All
order_status,,,
approved,0,2,2
canceled,75,550,625
created,0,5,5
delivered,96476,2,96478
invoiced,0,314,314
processing,0,301,301
shipped,1107,0,1107
unavailable,0,609,609
All,97658,1783,99441


order_delivered_customer_date


order_delivered_customer_date,False,True,All
order_status,,,
approved,0,2,2
canceled,6,619,625
created,0,5,5
delivered,96470,8,96478
invoiced,0,314,314
processing,0,301,301
shipped,0,1107,1107
unavailable,0,609,609
All,96476,2965,99441


In [6]:
def investigate_gap(df, earlier_col, later_col):
    """Rows where later_col is before earlier_col, and how big the gap is."""
    bad = df[df[later_col] < df[earlier_col]].copy()
    bad["gap_days"] = (bad[earlier_col] - bad[later_col]).dt.total_seconds() / 86400

    print(f"{later_col} before {earlier_col}: {len(bad)} rows\n")
    print("Gap in days:")
    print(bad["gap_days"].describe().round(2), "\n")

    print("Gap size buckets:")
    buckets = pd.cut(bad["gap_days"], bins=[0, 1, 7, 30, float("inf")],
                     labels=["under 1 day", "1-7 days", "7-30 days", "over 30 days"])
    print(buckets.value_counts().sort_index(), "\n")

    print("Order status:")
    print(bad["order_status"].value_counts(), "\n")

    print("Sample rows:")
    display(bad[["order_id", "order_status", earlier_col, later_col, "gap_days"]].head())
    return bad

# 1,359 rows: carrier handover before payment approval
gap_carrier_before_approval = investigate_gap(
    orders, "order_approved_at", "order_delivered_carrier_date")

# 23 rows: customer delivery before carrier handover (small enough to read in full)
gap_delivery_before_carrier = investigate_gap(
    orders, "order_delivered_carrier_date", "order_delivered_customer_date")

order_delivered_carrier_date before order_approved_at: 1359 rows

Gap in days:
count    1359.00
mean        1.03
std         4.80
min         0.00
25%         0.06
50%         0.72
75%         1.08
max       171.22
Name: gap_days, dtype: float64 

Gap size buckets:
gap_days
under 1 day     901
1-7 days        444
7-30 days        13
over 30 days      1
Name: count, dtype: int64 

Order status:
order_status
delivered    1350
shipped         9
Name: count, dtype: int64 

Sample rows:


,order_id,order_status,order_approved_at,order_delivered_carrier_date,gap_days
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-12 23:31:02,2018-06-11 14:54:00,1.359051
64,688052146432ef8253587b930b01a06d,delivered,2018-04-24 18:25:22,2018-04-23 19:19:14,0.962593
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-26 23:31:53,2018-07-24 12:57:00,2.440891
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-23 12:31:53,2018-07-23 12:24:00,0.005475
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-27 23:31:09,2018-07-24 14:03:00,3.394549


order_delivered_customer_date before order_delivered_carrier_date: 23 rows

Gap in days:
count    23.00
mean      3.27
std       3.72
min       0.02
25%       0.97
50%       1.66
75%       5.36
max      16.10
Name: gap_days, dtype: float64 

Gap size buckets:
gap_days
under 1 day      7
1-7 days        13
7-30 days        3
over 30 days     0
Name: count, dtype: int64 

Order status:
order_status
delivered    23
Name: count, dtype: int64 

Sample rows:


,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date,gap_days
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28 16:57:58,2017-07-25 19:32:56,2.892384
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-07 17:22:41,2017-07-06 14:27:51,1.121412
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-20 19:22:02,2017-07-19 14:13:28,1.214282
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10,6.009954
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-20 23:03:42,2017-07-20 18:52:41,0.174317


In [7]:
display(orders[(orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isnull()])

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [10]:
orders_clean = clean_orders(datasets["olist_orders_dataset"])
raw = datasets["olist_orders_dataset"]

print("rows raw/clean:", len(raw), len(orders_clean))
print("order_id unique:", orders_clean["order_id"].is_unique)
print("raw dtype untouched:", raw["order_purchase_timestamp"].dtype)   # should still be str

# nulls unchanged: 0, 160, 1783, 2965, 0
print(orders_clean[cols].isnull().sum())

# flag counts
print(orders_clean.filter(like="flag_").sum())

rows raw/clean: 99441 99441
order_id unique: True
raw dtype untouched: str
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
flag_carrier_before_approval    1359
flag_delivery_before_carrier      23
flag_delivered_missing_date       23
dtype: int64


In [12]:
from src.validation.data_quality import (
    check_missing_values, check_duplicates, check_unique_values,
    check_data_type, check_date_order,
)
name = "olist_orders_dataset"
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# check what dtype pandas gave the converted columns
print(orders_clean[timestamp_cols].dtypes, "\n")

results = []

# 1. data types: should now be clean
for col in timestamp_cols:
    results.append(check_data_type(orders_clean, name, col, "datetime64[ns]"))

# 2. date order: should STILL report 1,359 and 23, because dates are flagged, not fixed
results.append(check_date_order(orders_clean, name, "order_approved_at", "order_delivered_carrier_date"))
results.append(check_date_order(orders_clean, name, "order_delivered_carrier_date", "order_delivered_customer_date"))

# 3. missing values: should STILL report the three expected timestamp nulls
results.append(check_missing_values(orders_clean, name))

# 4. duplicates and key
results.append(check_duplicates(orders_clean, name))
results.append(check_unique_values(orders_clean, name, ["order_id"]))

revalidation = pd.concat(results, ignore_index=True)
print(revalidation.shape)
print(revalidation[["check", "column", "count", "message"]].to_string())

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object 

(5, 7)
            check                                                       column count                                                                          message
0      date_order              order_approved_at, order_delivered_carrier_date  1359            order_delivered_carrier_date is before order_approved_at in 1359 rows
1      date_order  order_delivered_carrier_date, order_delivered_customer_date    23  order_delivered_customer_date is before order_delivered_carrier_date in 23 rows
2  missing_values                                            order_approved_at   160                                    order_approved_at: 160 missing values (0.16%)
3  missing_values                                 order_delivered_carrier

In [13]:
payments = datasets["olist_order_payments_dataset"]

bad_inst = payments[payments["payment_installments"] < 1]
display(bad_inst)

# all payment rows belonging to those orders
display(payments[payments["order_id"].isin(bad_inst["order_id"])].sort_values(["order_id", "payment_sequential"]))

# the near-zero payment values, and overlap with the group above
bad_val = payments[payments["payment_value"] < 0.01]
display(bad_val)
print("overlap:", set(bad_inst.index) & set(bad_val.index))

# value counts to spot placeholder types like not_defined
display(payments["payment_type"].value_counts())

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


overlap: set()


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [14]:
# 1. orders whose payment_sequential doesn't run 1..n
g = payments.groupby("order_id")["payment_sequential"].agg(["min", "max", "count"])
gaps = g[(g["min"] != 1) | (g["max"] != g["count"])]
print("orders with a sequence gap:", len(gaps))
display(gaps.head(10))

# 2. status of the not_defined orders
nd = payments[payments["payment_type"] == "not_defined"]["order_id"]
orders = datasets["olist_orders_dataset"]
display(orders[orders["order_id"].isin(nd)][["order_id", "order_status"]])

orders with a sequence gap: 80


,min,max,count
order_id,,,
00ac05fe0fc047c54418098eb64e3aaa,2,2,1
056c68d093c100017aab1f00f260705c,2,2,1
0668d086b3ae41adde3aed3dacbc8fae,2,2,1
0a7d898c6305101e69e9f5a05cd0130d,2,2,1
0d0e88a418636d40ea780339701db49c,2,2,1
0fdb6f73fab7005c84d4a43d8a93682b,2,2,1
159da9914b51de617fe80162939965c5,2,2,1
166b01d0fce75a595b92ad9e84e0843b,2,2,1
1896e8b8fd196151559b02e31e98ce89,2,2,1


,order_id,order_status
1130,00b1cb0320190ca0daa2c88b35206009,canceled
39919,4637ca194b6387e2d538dc89b124b0ee,canceled
40235,c8c528189310eaa44a745b8d9d26908b,canceled


In [15]:
from src.cleaning.payments import clean_payments
raw = datasets["olist_order_payments_dataset"]
payments_clean = clean_payments(raw)

print("rows raw/clean:", len(raw), len(payments_clean))      # 103886 103886
print(payments_clean.filter(like="flag_").sum())             # 2 and 80
print("raw has no flags:", "flag_zero_installments" not in raw.columns)   # True

rows raw/clean: 103886 103886
flag_zero_installments         2
flag_missing_first_payment    82
dtype: int64
raw has no flags: True


In [16]:
flagged = payments_clean[payments_clean["flag_missing_first_payment"]]
print("flagged orders:", flagged["order_id"].nunique())
print(flagged.groupby("order_id").size().value_counts())   # rows per flagged order

# gap orders that are NOT flagged (gap in the middle)
gap_orders = gaps.index
not_flagged = set(gap_orders) - set(flagged["order_id"])
print("gap orders not flagged:", len(not_flagged))

flagged orders: 80
1    78
2     2
Name: count, dtype: int64
gap orders not flagged: 0


In [17]:
products = datasets["olist_products_dataset"]
trans = datasets["product_category_name_translation"]

# issue 3: weight below 1 g
display(products[products["product_weight_g"] < 1])

# issue 4: categories products use that the translation table doesn't have
missing_cats = products[
    ~products["product_category_name"].isin(trans["product_category_name"])
    & products["product_category_name"].notnull()
]
print(missing_cats["product_category_name"].value_counts())

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64


In [18]:
from src.cleaning.category_translation import clean_category_translation

raw_trans = datasets["product_category_name_translation"]
trans_clean = clean_category_translation(raw_trans)

print("rows raw/clean:", len(raw_trans), len(trans_clean))   # 71 73
print("raw untouched:", len(raw_trans) == 71)                # True

# the 13 products should now all have a translation
missing = products[
    ~products["product_category_name"].isin(trans_clean["product_category_name"])
    & products["product_category_name"].notnull()
]
print("untranslated products:", len(missing))                # 0

rows raw/clean: 71 73
raw untouched: True
untranslated products: 0


In [19]:
display(products[products["product_weight_g"] < 1])

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


In [20]:
from src.cleaning.products import clean_products
from src.cleaning.category_translation import clean_category_translation

raw = datasets["olist_products_dataset"]
products_clean = clean_products(raw)
trans_clean = clean_category_translation(datasets["product_category_name_translation"])

# nothing lost, raw untouched
print("rows raw/clean:", len(raw), len(products_clean))                       # 32951 32951
print("raw still has old names:", "product_name_lenght" in raw.columns)       # True
print("raw weight zeros still there:", (raw["product_weight_g"] == 0).sum())  # 4

# the fixes worked
print("zero-weight flag:", products_clean["flag_zero_weight"].sum())          # 4
print("weight nulls:", products_clean["product_weight_g"].isnull().sum())     # 6 (2 + 4)
print("category nulls:", products_clean["product_category_name"].isnull().sum())    # 0
print("unknown category:", (products_clean["product_category_name"] == "unknown").sum())  # 610
print(products_clean.columns.tolist())

# translation table and integrity
print("translation rows:", len(trans_clean))                                  # 74
print("untranslated products:", (~products_clean["product_category_name"].isin(trans_clean["product_category_name"])).sum())  # 0

rows raw/clean: 32951 32951
raw still has old names: True
raw weight zeros still there: 4
zero-weight flag: 4
weight nulls: 6
category nulls: 0
unknown category: 610
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'flag_zero_weight']
translation rows: 74
untranslated products: 0


In [22]:
from src.validation.data_quality import (
    check_missing_values, check_duplicates, check_unique_values, check_value_range)
name = "olist_products_dataset"
display(pd.concat([
    check_missing_values(products_clean, name),
    check_value_range(products_clean, name, "product_weight_g", min_value=1),
    check_duplicates(products_clean, name),
    check_unique_values(products_clean, name, ["product_id"]),
], ignore_index=True))

,table,check,column,count,percent,severity,message
0,olist_products_dataset,missing_values,product_name_length,610,1.85,warning,product_name_length: 610 missing values (1.85%)
1,olist_products_dataset,missing_values,product_description_length,610,1.85,warning,product_description_length: 610 missing values...
2,olist_products_dataset,missing_values,product_photos_qty,610,1.85,warning,product_photos_qty: 610 missing values (1.85%)
3,olist_products_dataset,missing_values,product_weight_g,6,0.02,warning,product_weight_g: 6 missing values (0.02%)
4,olist_products_dataset,missing_values,product_length_cm,2,0.01,warning,product_length_cm: 2 missing values (0.01%)
5,olist_products_dataset,missing_values,product_height_cm,2,0.01,warning,product_height_cm: 2 missing values (0.01%)
6,olist_products_dataset,missing_values,product_width_cm,2,0.01,warning,product_width_cm: 2 missing values (0.01%)


In [23]:
sellers = datasets["olist_sellers_dataset"]
s = sellers["seller_city"]
norm = (s.str.strip().str.lower()
        .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii"))

variants = s.groupby(norm).nunique()
bad = variants[variants > 1].index
display(s[norm.isin(bad)].value_counts().sort_index())

seller_city
santa barbara d oeste      1
santa barbara d´oeste      2
sao paulo                694
são paulo                 1
Name: count, dtype: int64

In [26]:
from src.cleaning.sellers import clean_sellers
from src.validation.data_quality import (
    check_missing_values, check_duplicates, check_unique_values, check_data_type, check_categorical_values)
raw = datasets["olist_sellers_dataset"]
sellers_clean = clean_sellers(raw)
zip_col = "seller_zip_code_prefix"
name = "olist_sellers_dataset"

# nothing lost, raw untouched
print("rows raw/clean:", len(raw), len(sellers_clean))                      # 3095 3095
print("raw zip dtype:", raw[zip_col].dtype)                                 # int64
print("distinct zips raw/clean:", raw[zip_col].nunique(), sellers_clean[zip_col].nunique())
print("zip lengths:", sellers_clean[zip_col].str.len().value_counts().to_dict())   # only 5
print("round trip matches raw:", (sellers_clean[zip_col].astype(int) == raw[zip_col]).all())

# city changes: only the variants should have changed
changed = raw["seller_city"] != sellers_clean["seller_city"]
print("city rows changed:", changed.sum())
display(pd.DataFrame({"raw": raw.loc[changed, "seller_city"],
                      "clean": sellers_clean.loc[changed, "seller_city"]}).value_counts())

# re-validate
display(pd.concat([
    check_data_type(sellers_clean, name, zip_col, str),
    check_categorical_values(sellers_clean, name, "seller_city"),
    check_missing_values(sellers_clean, name),
    check_duplicates(sellers_clean, name),
    check_unique_values(sellers_clean, name, ["seller_id"]),
], ignore_index=True))

rows raw/clean: 3095 3095
raw zip dtype: int64
distinct zips raw/clean: 2246 2246
zip lengths: {5: 3095}
round trip matches raw: True
city rows changed: 3


raw                    clean                
santa barbara d´oeste  santa barbara d oeste    2
são paulo             sao paulo                1
Name: count, dtype: int64

,table,check,column,count,percent,severity,message


In [27]:
items = datasets["olist_order_items_dataset"]
zero = items[items["freight_value"] == 0]
print("zero-freight rows:", len(zero))

# 1. concentrated in a few sellers?
print(zero["seller_id"].value_counts().head(10))

# 2. do they have normal prices?
print(zero["price"].describe().round(2))

# 3. for orders with several items, do other items in the same order pay freight?
multi = items[items["order_id"].isin(zero["order_id"])]
print(multi.groupby("order_id")["freight_value"].agg(["count", "min", "max"]).head(10))

zero-freight rows: 383
seller_id
7d13fca15225358621be4086e1eb0964    158
955fee9216a65b617aa5c0531780ce60     99
1f50f920176fa81dab994f9023523100     56
4869f7a5dfa277a7dca6462dcf3b52b2     56
37be5a7c751166fbc5f8ccba4119e043      9
c826c40d7b19f62a09e2d7c5e7295ee2      2
bc2ac6b95e1accce9858528ee566c17e      1
cc419e0650a3c5ba77189a1882b7556a      1
8581055ce74af1daba164fdbd55a40de      1
Name: count, dtype: int64
count    383.0
mean      98.6
std       50.0
min       53.9
25%       69.9
50%       99.9
75%      106.9
max      712.9
Name: price, dtype: float64
                                  count  min  max
order_id                                         
00404fa7a687c8c44ca69d42695aae73      1  0.0  0.0
00a870c6c06346e85335524935c600c0      1  0.0  0.0
011c899816ea29773525bd3322dbb6aa      1  0.0  0.0
012b3f6ab7776a8ab3443a4ad7bef2e6      2  0.0  0.0
04105b54650921ca3246f52e6f175f46      1  0.0  0.0
0517a3e68dac3308995edca2144db36e      1  0.0  0.0
061ba2e2d7544790b6ed6b5b4dd9278c 

In [28]:
display(items.nlargest(10, "price")[["order_id", "product_id", "seller_id", "price", "freight_value"]])

,order_id,product_id,seller_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,69c590f7ffc7bf8db97190b6cb6ed62e,80ceebb4ee9b31afb6c6a916a574a1e2,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1bdf5e6731585cf01aa8169c7028d6ad,ee27a8f15b1dded4d213a468ba4eb391,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,a6492cc69376c469ab6f61d8f44de961,59417c56835dd8e2e72f91f809cd4092,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,c3ed642d592594bb648ff4a04cee2747,59417c56835dd8e2e72f91f809cd4092,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,259037a6a41845e455183f89c5035f18,c72de06d72748d1a0dfb2125be43ba63,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,a1beef8f3992dbd4cd8726796aa69c53,512d298ac2a96d1931b6bd30aa21f61d,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,6cdf8fc1d741c76586d8b6b15e9eef30,ed4acab38528488b65a9a9c603ff024a,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,dd113cb02b2af9c8e5787e8f1f0722f6,821fb029fc6e495ca4f08a35d51e53a5,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,6902c1962dd19d540807d0ab8fade5c6,fa1c13f2614d7b5c4749cbc52fecda94,3999.90,17.01


In [29]:
from src.cleaning.order_items import clean_order_items

raw = datasets["olist_order_items_dataset"]
items_clean = clean_order_items(raw)
name = "olist_order_items_dataset"

print("rows raw/clean:", len(raw), len(items_clean))                        # 112650 112650
print("raw dtype untouched:", raw["shipping_limit_date"].dtype)             # str
print("clean dtype:", items_clean["shipping_limit_date"].dtype)             # datetime64[us]
print("nulls:", items_clean["shipping_limit_date"].isnull().sum())          # 0
print("other columns identical:", raw.drop(columns="shipping_limit_date").equals(
    items_clean.drop(columns="shipping_limit_date")))                       # True

display(pd.concat([
    check_data_type(items_clean, name, "shipping_limit_date", "datetime64[ns]"),
    check_unique_values(items_clean, name, ["order_id", "order_item_id"]),
    check_duplicates(items_clean, name),
    check_missing_values(items_clean, name),
    check_value_range(items_clean, name, "freight_value", min_value=0.01),
], ignore_index=True))

rows raw/clean: 112650 112650
raw dtype untouched: str
clean dtype: datetime64[us]
nulls: 0
other columns identical: True


,table,check,column,count,percent,severity,message
0,olist_order_items_dataset,invalid_range,freight_value,383,0.34,error,"freight_value: 383 values outside [0.01, None]"


In [30]:
reviews = datasets["olist_order_reviews_dataset"]

# 1. repeated order_id: what do the rows look like?
dup_orders = reviews[reviews.duplicated("order_id", keep=False)].sort_values(["order_id", "review_answer_timestamp"])
print("rows involved:", len(dup_orders), "| orders:", dup_orders["order_id"].nunique())
display(dup_orders[["order_id", "review_id", "review_score", "review_creation_date", "review_answer_timestamp"]].head(10))

# do the repeated reviews for one order disagree on the score?
spread = dup_orders.groupby("order_id")["review_score"].nunique()
print("orders whose repeated reviews have different scores:", (spread > 1).sum(), "of", len(spread))

# 2. repeated review_id: same review attached to different orders?
dup_ids = reviews[reviews.duplicated("review_id", keep=False)]
print("orders per repeated review_id:")
print(dup_ids.groupby("review_id")["order_id"].nunique().value_counts())

# are the repeated review_id rows identical apart from order_id?
print("repeated review_id rows that are also full-row duplicates:",
      dup_ids.duplicated(keep=False).sum())

rows involved: 1098 | orders: 547


,order_id,review_id,review_score,review_creation_date,review_answer_timestamp
22423,0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,2017-08-25 00:00:00,2017-08-29 21:45:57
25612,0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,2017-08-29 00:00:00,2017-08-30 01:59:12
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,2018-03-04 00:00:00,2018-03-05 17:02:00
854,0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5,2017-12-30 00:00:00,2018-01-02 10:54:47
89888,02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3,2018-03-21 00:00:00,2018-03-22 01:32:08
17582,02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1,2018-03-29 00:00:00,2018-03-30 03:16:19
37911,029863af4b968de1e5d6a82782e662f5,04d945e95c788a3aa1ffbee42105637b,5,2017-07-14 00:00:00,2017-07-17 13:58:06
55137,029863af4b968de1e5d6a82782e662f5,61fe4e7d1ae801bbe169eb67b86c6eda,4,2017-07-19 00:00:00,2017-07-20 12:06:11


orders whose repeated reviews have different scores: 202 of 547
orders per repeated review_id:
order_id
2    764
3     25
Name: count, dtype: int64
repeated review_id rows that are also full-row duplicates: 0


In [31]:
from src.cleaning.order_reviews import clean_order_reviews

raw = datasets["olist_order_reviews_dataset"]
reviews_clean = clean_order_reviews(raw)
name = "olist_order_reviews_dataset"

print("rows raw/clean:", len(raw), len(reviews_clean))            # 99224 98673
print("rows dropped:", len(raw) - len(reviews_clean))             # 551
print("orders raw/clean:", raw["order_id"].nunique(), reviews_clean["order_id"].nunique())  # 98673 98673
print("raw dtype untouched:", raw["review_creation_date"].dtype)  # str

# the kept review must be the latest one for every order
latest = pd.to_datetime(raw["review_answer_timestamp"]).groupby(raw["order_id"]).max()
kept = reviews_clean.set_index("order_id")["review_answer_timestamp"]
print("kept review is the latest for every order:", (kept == latest.loc[kept.index]).all())

display(pd.concat([
    check_data_type(reviews_clean, name, "review_creation_date", "datetime64[ns]"),
    check_data_type(reviews_clean, name, "review_answer_timestamp", "datetime64[ns]"),
    check_unique_values(reviews_clean, name, ["order_id"]),
    check_unique_values(reviews_clean, name, ["review_id"]),
    check_missing_values(reviews_clean, name),
    check_duplicates(reviews_clean, name),
    check_value_range(reviews_clean, name, "review_score", min_value=1, max_value=5),
], ignore_index=True))

rows raw/clean: 99224 98673
rows dropped: 551
orders raw/clean: 98673 98673
raw dtype untouched: str
kept review is the latest for every order: True


,table,check,column,count,percent,severity,message
0,olist_order_reviews_dataset,unique_values,review_id,578,0.59,error,review_id is not unique: 578 repeated rows (0....
1,olist_order_reviews_dataset,missing_values,review_comment_title,87123,88.29,warning,review_comment_title: 87123 missing values (88...
2,olist_order_reviews_dataset,missing_values,review_comment_message,57898,58.68,warning,review_comment_message: 57898 missing values (...


In [32]:
geo = datasets["olist_geolocation_dataset"]

outside = geo[
    (geo["geolocation_lat"] < -33.8) | (geo["geolocation_lat"] > 5.3)
    | (geo["geolocation_lng"] < -73.9) | (geo["geolocation_lng"] > -34.8)
]
print("rows outside Brazil:", len(outside))
print("distinct zip prefixes affected:", outside["geolocation_zip_code_prefix"].nunique())
print(outside["geolocation_state"].value_counts().head())
display(outside.head(10))

# do those zip prefixes have other, normal points we could rely on instead?
other_points = geo[geo["geolocation_zip_code_prefix"].isin(outside["geolocation_zip_code_prefix"])]
print("rows for those prefixes, in total:", len(other_points))

rows outside Brazil: 47
distinct zip prefixes affected: 23
geolocation_state
PE    11
PA     7
PB     6
RJ     5
BA     4
Name: count, dtype: int64


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


rows for those prefixes, in total: 436


In [33]:
from src.cleaning.geolocation import clean_geolocation
from src.cleaning.customers import clean_customers
from src.cleaning.sellers import clean_sellers

raw = datasets["olist_geolocation_dataset"]
geo_clean = clean_geolocation(raw)
name = "olist_geolocation_dataset"
z = "geolocation_zip_code_prefix"

# raw untouched, and what changed
print("rows raw/clean:", len(raw), len(geo_clean))
print("raw zip dtype:", raw[z].dtype)                                    # int64
print("exact duplicates in raw:", raw.duplicated().sum())                # 261831

# prefixes: did any lose all their points?
raw_prefixes = set(raw[z].astype(str).str.zfill(5))
print("prefixes raw/clean:", len(raw_prefixes), geo_clean[z].nunique())
print("prefixes lost:", sorted(raw_prefixes - set(geo_clean[z])))

# the cleaned table
print("one row per prefix:", geo_clean[z].is_unique)
print("zip lengths:", geo_clean[z].str.len().value_counts().to_dict())   # only 5
print("nulls:", geo_clean.isnull().sum().to_dict())
print(geo_clean[["geolocation_lat", "geolocation_lng"]].describe().round(2))

# re-validate
display(pd.concat([
    check_data_type(geo_clean, name, z, str),
    check_duplicates(geo_clean, name),
    check_unique_values(geo_clean, name, [z]),
    check_value_range(geo_clean, name, "geolocation_lat", min_value=-33.8, max_value=5.3),
    check_value_range(geo_clean, name, "geolocation_lng", min_value=-73.9, max_value=-34.8),
    check_categorical_values(geo_clean, name, "geolocation_city"),
    check_missing_values(geo_clean, name),
], ignore_index=True))

# integrity: customers and sellers against the cleaned geolocation
cust = clean_customers(datasets["olist_customers_dataset"])
sell = clean_sellers(datasets["olist_sellers_dataset"])
print("customer zips not in geolocation:", (~cust["customer_zip_code_prefix"].isin(geo_clean[z])).sum())  # about 278
print("seller zips not in geolocation:", (~sell["seller_zip_code_prefix"].isin(geo_clean[z])).sum())      # about 7

rows raw/clean: 1000163 19010
raw zip dtype: int64
exact duplicates in raw: 261831
prefixes raw/clean: 19015 19010
prefixes lost: ['18243', '53990', '78131', '83252', '95130']
one row per prefix: True
zip lengths: {5: 19010}
nulls: {'geolocation_zip_code_prefix': 0, 'geolocation_lat': 0, 'geolocation_lng': 0, 'geolocation_city': 0, 'geolocation_state': 0}
       geolocation_lat  geolocation_lng
count         19010.00         19010.00
mean            -19.08           -46.08
std               7.24             5.20
min             -33.69           -72.91
25%             -23.56           -49.01
50%             -22.43           -46.63
75%             -15.62           -43.25
max               4.48           -34.80


,table,check,column,count,percent,severity,message


customer zips not in geolocation: 279
seller zips not in geolocation: 7


In [3]:
from src.cleaning.customers import clean_customers
from src.cleaning.orders import clean_orders
from src.cleaning.payments import clean_payments
from src.cleaning.products import clean_products
from src.cleaning.sellers import clean_sellers
from src.cleaning.order_items import clean_order_items
from src.cleaning.order_reviews import clean_order_reviews
from src.cleaning.geolocation import clean_geolocation
from src.cleaning.category_translation import clean_category_translation

cleaners = {
    "olist_customers_dataset": clean_customers,
    "olist_orders_dataset": clean_orders,
    "olist_order_payments_dataset": clean_payments,
    "olist_products_dataset": clean_products,
    "olist_sellers_dataset": clean_sellers,
    "olist_order_items_dataset": clean_order_items,
    "olist_order_reviews_dataset": clean_order_reviews,
    "olist_geolocation_dataset": clean_geolocation,
    "product_category_name_translation": clean_category_translation,
}

processed_dir = root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

cleaned = {}
for name, clean_fn in cleaners.items():
    cleaned[name] = clean_fn(datasets[name])
    cleaned[name].to_parquet(processed_dir / f"{name}.parquet", index=False)
    print(f"{name}: {len(datasets[name]):>9,} -> {len(cleaned[name]):>9,} rows saved")

olist_customers_dataset:    99,441 ->    99,441 rows saved
olist_orders_dataset:    99,441 ->    99,441 rows saved
olist_order_payments_dataset:   103,886 ->   103,886 rows saved
olist_products_dataset:    32,951 ->    32,951 rows saved
olist_sellers_dataset:     3,095 ->     3,095 rows saved
olist_order_items_dataset:   112,650 ->   112,650 rows saved
olist_order_reviews_dataset:    99,224 ->    98,673 rows saved
olist_geolocation_dataset: 1,000,163 ->    19,010 rows saved
product_category_name_translation:        71 ->        74 rows saved


In [4]:
rows = []
for name, clean in cleaned.items():
    raw = datasets[name]
    rows.append({
        "table": name,
        "rows_raw": len(raw),
        "rows_clean": len(clean),
        "cols_added": sorted(set(clean.columns) - set(raw.columns)),
        "cols_removed": sorted(set(raw.columns) - set(clean.columns)),
    })
display(pd.DataFrame(rows))

,table,rows_raw,rows_clean,cols_added,cols_removed
0,olist_customers_dataset,99441,99441,[],[]
1,olist_orders_dataset,99441,99441,"[flag_carrier_before_approval, flag_delivered_...",[]
2,olist_order_payments_dataset,103886,103886,"[flag_missing_first_payment, flag_zero_install...",[]
3,olist_products_dataset,32951,32951,"[flag_zero_weight, product_description_length,...","[product_description_lenght, product_name_lenght]"
4,olist_sellers_dataset,3095,3095,[],[]
5,olist_order_items_dataset,112650,112650,[],[]
6,olist_order_reviews_dataset,99224,98673,[has_comment],[]
7,olist_geolocation_dataset,1000163,19010,[],[]
8,product_category_name_translation,71,74,[],[]


In [5]:
for name, clean in cleaned.items():
    raw = datasets[name]
    shared = [c for c in raw.columns if c in clean.columns]
    diff = clean[shared].isnull().sum() - raw[shared].isnull().sum()
    diff = diff[diff != 0]
    print(name, diff.to_dict() if len(diff) else "no null changes")

olist_customers_dataset no null changes
olist_orders_dataset no null changes
olist_order_payments_dataset no null changes
olist_products_dataset {'product_category_name': -610, 'product_weight_g': 4}
olist_sellers_dataset no null changes
olist_order_items_dataset no null changes
olist_order_reviews_dataset {'review_comment_title': -533, 'review_comment_message': -349}
olist_geolocation_dataset no null changes
product_category_name_translation no null changes


In [6]:
raw_items, raw_pay = datasets["olist_order_items_dataset"], datasets["olist_order_payments_dataset"]
cl_items, cl_pay = cleaned["olist_order_items_dataset"], cleaned["olist_order_payments_dataset"]

print("price sum equal:", raw_items["price"].sum() == cl_items["price"].sum())
print("freight sum equal:", raw_items["freight_value"].sum() == cl_items["freight_value"].sum())
print("payment sum equal:", raw_pay["payment_value"].sum() == cl_pay["payment_value"].sum())

# reviews: the kept rows keep their scores
raw_rev, cl_rev = datasets["olist_order_reviews_dataset"], cleaned["olist_order_reviews_dataset"]
print("review score mean raw/clean:", round(raw_rev["review_score"].mean(), 3), round(cl_rev["review_score"].mean(), 3))

# orders: every order still there
print("orders ids identical:", set(datasets["olist_orders_dataset"]["order_id"]) == set(cleaned["olist_orders_dataset"]["order_id"]))

price sum equal: True
freight sum equal: True
payment sum equal: True
review score mean raw/clean: 4.086 4.086
orders ids identical: True
